# FoldMason Toolkit

![FoldMason](https://proto-bio.github.io/proto-assets/images/tool/foldmason/hero.png)

This notebook demonstrates the two FoldMason tools:

1. **`foldmason-msa`** — multiple structure alignment (remote default; tuneable local mode)
2. **`foldmason-score-msa`** — score an existing MSA with average + per-column LDDT (local-only)

Reference: Gilchrist et al., *Science* 2026 ([DOI](https://doi.org/10.1126/science.ads6733)).

In [ ]:
from proto_tools import Structure
from proto_tools.tools.structure_alignment import (
    FoldmasonMSAConfig, FoldmasonMSAInput, run_foldmason_msa,
    FoldmasonScoreMSAConfig, FoldmasonScoreMSAInput, run_foldmason_score_msa,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Step 1: fetch a TIM-barrel structural family

We'll align three TIM-barrel triosephosphate isomerases from different organisms — a classic example of structurally similar enzymes spanning broad sequence divergence.

In [ ]:
ids = ["1TIM", "8TIM", "1TPF"]
structures = [Structure.from_rcsb(pdb_id) for pdb_id in ids]
print(f"Fetched {len(structures)} TIM-barrel structures: {ids}")

## Step 2: align with `foldmason-msa` (local)

Runs the FoldMason CLI in an isolated standalone env. We use local mode here so the next step (`foldmason-score-msa`, which is local-only) can resolve each MSA row to its structure — the public server's MSA naming differs from the local createdb naming for multi-chain inputs.

In [ ]:
# Input fields for run_foldmason_msa
display_api_reference("foldmason", "input", "run_foldmason_msa")

In [ ]:
# Config fields for run_foldmason_msa
display_api_reference("foldmason", "config", "run_foldmason_msa")

In [ ]:
# Output fields for run_foldmason_msa
display_api_reference("foldmason", "output", "run_foldmason_msa")

In [ ]:
msa_out = run_foldmason_msa(
    FoldmasonMSAInput(structures=structures, structure_ids=ids),
    FoldmasonMSAConfig(search_mode="local", num_threads=2),
)
print(f"Aligned {msa_out.num_sequences} structures across {msa_out.alignment_length} columns")
print(f"Guide tree: {msa_out.newick_tree.strip()}")
print()
for line in msa_out.aa_msa_fasta.splitlines()[:6]:
    print(line[:100] + (" ..." if len(line) > 100 else ""))

## Step 3: score the alignment with `foldmason-score-msa` (local)

Computes per-column and average LDDT scores against the input structures. Useful as a quality metric or downstream design constraint. Local-only — `msa2lddt` is not exposed by the public server.

In [ ]:
# Input fields for run_foldmason_score_msa
display_api_reference("foldmason", "input", "run_foldmason_score_msa")

In [ ]:
# Config fields for run_foldmason_score_msa
display_api_reference("foldmason", "config", "run_foldmason_score_msa")

In [ ]:
# Output fields for run_foldmason_score_msa
display_api_reference("foldmason", "output", "run_foldmason_score_msa")

In [ ]:
score_out = run_foldmason_score_msa(
    FoldmasonScoreMSAInput(structures=structures, structure_ids=ids, msa=msa_out.aa_msa_fasta),
    FoldmasonScoreMSAConfig(num_threads=2),
)
print(f"Average MSA LDDT: {score_out.average_lddt:.3f}")
print(f"Columns considered: {score_out.columns_considered}/{score_out.alignment_length}")
print(f"First 10 column scores: {[round(s, 2) for s in score_out.column_scores[:10]]}")